# Chapter 8: Gaussian Processes. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/08-gp.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/08-gp.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/08-gp.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("laGP", "MASS")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

## Exercise 8.6: GP Prior Sampling

Consider a Gaussian Process with mean function $m(x) = 0$ and squared exponential kernel:
$$
k(x, x') = \sigma^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right)
$$

a) For $\sigma^2 = 1$ and $\ell = 1$, calculate the covariance matrix $K$ for points $x = (0, 1, 2)^T$.

b) Sample 5 functions from this GP prior at points $x = (0, 0.5, 1, 1.5, 2, 2.5, 3)$.

c) How does changing $\ell$ to 0.5 affect the sampled functions?

d) How does changing $\sigma^2$ to 4 affect the sampled functions?

## Exercise 8.7: GP Posterior Prediction

You observe the following data points:

| x | y |
| -- | -- |
| 0 | 1.0 |
| 2 | 0.5 |
| 4 | 2.0 |

Assume a GP prior with $m(x) = 0$, squared exponential kernel with $\sigma^2 = 1$, $\ell = 1$, and observation noise $\sigma_n^2 = 0.1$. Because real observations include noise, this exercise uses the noisy-GP posterior obtained from Equation 8.1 and Equation 8.2 by replacing $\mathbf{K}$ with $\mathbf{K} + \sigma_n^2\mathbf{I}$ wherever it appears, equivalent to observing $y_i = f(x_i) + \epsilon_i$ with $\epsilon_i \overset{\text{i.i.d.}}{\sim} \mathcal{N}(0,\sigma_n^2)$.

a) Write the posterior mean formula $\mu_*$ for predictions at new points.

b) Calculate the posterior mean and variance at $x_* = 1$.

c) Calculate the posterior mean and variance at $x_* = 3$.

d) Plot the posterior mean and 95% credible interval for $x \in [0, 5]$.

## Exercise 8.8: Log Marginal Likelihood

Using the data from the GP Posterior Prediction exercise above ($X = (0, 2, 4)$, $y = (1.0, 0.5, 2.0)$), the log marginal likelihood for a (noiseless) GP is:
$$
\log p(y \mid X, \theta) = -\frac{1}{2}y^T K^{-1} y - \frac{1}{2}\log|K| - \frac{n}{2}\log(2\pi)
$$
where $\theta = (\sigma^2, \ell)$ are the kernel hyperparameters. This uses the noiseless log marginal likelihood matching Equation 8.1 and Equation 8.2 as derived in the chapter, which generalizes to the noisy-GP posterior of the previous exercise by replacing $\mathbf{K}$ with $\mathbf{K}+\sigma_n^2\mathbf{I}$ throughout and extending $\theta$ to $(\sigma^2,\ell,\sigma_n^2)$.

a) Calculate the log marginal likelihood for $(\sigma^2 = 1, \ell = 1)$.

b) Calculate it for $(\sigma^2 = 1, \ell = 0.5)$.

c) Which hyperparameters are better supported by the data?

d) Interpret each term in the log marginal likelihood formula.

## Exercise 8.9: RBF versus Matern: Sample-Path Roughness

The squared exponential kernel is infinitely differentiable, which forces every function sampled from a GP with this kernel to be extremely smooth. A commonly used alternative is the Matern family of kernels, which controls smoothness directly through a parameter. The Matern-5/2 kernel, with length scale $\ell$ and signal variance $\sigma^2$, is
$$
k_{5/2}(x,x') = \sigma^2\left(1+\frac{\sqrt5\,r}{\ell}+\frac{5r^2}{3\ell^2}\right)\exp\left(-\frac{\sqrt5\,r}{\ell}\right), \qquad r = |x-x'|,
$$
and produces sample paths that are twice, but not infinitely, differentiable: rougher than the squared exponential kernel's paths at small scales.

a) On a grid of $n=200$ points over $[0,10]$, build the covariance matrices for the squared exponential kernel and the Matern-5/2 kernel above, both with $\sigma^2=1$, $\ell=1$.

b) Using the same underlying standard normal draws for both kernels, so that any difference between the two sets of sample paths is due to the kernel alone and not to randomness, draw 5 sample paths from each GP prior and plot them.

c) Compute a roughness statistic, such as the sum of squared successive differences $\sum_i(f(x_{i+1})-f(x_i))^2$ or the number of local extrema, for each path, and report the average over the 5 paths for each kernel.

d) Which kernel produces rougher paths, and does the direction of the difference match what the differentiability of the two kernels predicts?

## Exercise 8.10: GP Regression on the Motorcycle Data

The chapter's motorcycle accident example (Example 8.3) fits a GP to the full `MASS::mcycle` dataset using the `laGP` package. Here you will implement the GP posterior directly with matrix algebra, as in Equation 8.1 and Equation 8.2, and use it to fill in a deliberately held-out gap in the data.

a) Load `MASS::mcycle` and remove all observations with `times` in $(25, 30]$; call the remaining data the training set and the removed points the held-out test set.

b) Using a squared exponential kernel, estimate the signal variance $\sigma^2$, length scale $\ell$, and noise variance $\sigma_n^2$ by maximizing the log marginal likelihood on the training set only, following the same MLE approach as Example 8.2 but now including $\sigma_n^2$ as in the noisy-GP posterior of Exercise 8.4 (replace $\mathbf{K}$ with $\mathbf{K}+\sigma_n^2\mathbf{I}$ in Equation 8.1 and Equation 8.2). It is numerically convenient to standardize `accel` (subtract the training mean, divide by the training standard deviation) before fitting, and to transform predictions back to the original scale afterward.

c) Predict the posterior mean and standard deviation at the held-out times, and plot them (mean $\pm$ 2 standard deviations) against the true held-out observations.

d) Compare the root-mean-squared prediction error of the GP on the held-out points to that of a cubic polynomial regression fit on the same training data.